In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Delta_Project").getOrCreate()

data = [
    (1, "C001", "Laptop", 50000),
    (2, "C002", "Mobile", 15000),
    (3, "C003", "Tablet", 20000),
    (4, "C004", "Laptop", 55000)
]

columns = ["id", "customer_id", "product", "amount"]

In [0]:
df = spark.createDataFrame(data, columns)

** TASK 1: Create Delta Table**

In [0]:
# Save to a folder path
df.write.format("delta") \
  .mode("overwrite") \
  .save("/Volumes/workspace/default/day6_session/delta_table_data")

In [0]:
# TASK 2: Insert New Data
new_data=[(5, "C005", "Camera", 30000)]
new_df=spark.createDataFrame(new_data,columns)
new_df.write.format("delta")\
    .mode("append")\
     .save("/Volumes/workspace/default/day6_session/delta_table_data")

In [0]:
#task3: update Date
from delta.tables import DeltaTable
delta_table=DeltaTable.forPath(spark,"/Volumes/workspace/default/day6_session/delta_table_data")
delta_table.update(
    condition="id=2",
    set={"amount+":"8500"}
)

DataFrame[num_affected_rows: bigint]

In [0]:
#task4: Delete data
delta_table.delete("id=1")

In [0]:
# task5: Merge (incremental load)
updates = [
    (3, "C003", "Tablet", 22000),  # This will update the existing id=3
    (6, "C006", "Watch", 8000)     # This will insert a new record
]

updates_df = spark.createDataFrame(updates, columns)

delta_table.alias("target").merge(
    updates_df.alias("source"),
    "target.id = source.id"
).whenMatchedUpdate(set = {
    "product": "source.product",
    "amount": "source.amount"
}).whenNotMatchedInsert(values = {
    "id": "source.id",
    "customer_id": "source.customer_id",
    "product": "source.product",
    "amount": "source.amount"
}).execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
# task6: schema evolution
# add new column: country
new_data = [(7, "C007", "Tablet", 10000, "India")]
new_columns = ["id", "customer_id", "product", "amount", "country"]

df_new = spark.createDataFrame(new_data, new_columns)

df_new.write.format("delta") \
  .mode("append") \
  .option("mergeSchema", "true") \
  .save("/Volumes/workspace/default/day6_session/delta_table_data") # Added leading /

In [0]:
#task7: Time Travel
delta_table.history().display()

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
10,2026-04-14T16:11:21.000Z,73664067398892,tangallarohith5@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(3132550639151481),b2a370b1-6904-41b6-8f8c-7c9290504e34,0414-150555-h8g4gkkc-v2n,9,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1435)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
9,2026-04-14T15:59:07.000Z,73664067398892,tangallarohith5@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(3132550639151481),e0fe2482-3a3d-4062-b52c-a95bad3184c6,0414-150555-h8g4gkkc-v2n,8,SnapshotIsolation,false,"Map(numRemovedFiles -> 3, numRemovedBytes -> 3849, p25FileSize -> 1395, numDeletionVectorsRemoved -> 1, minFileSize -> 1395, numAddedFiles -> 1, maxFileSize -> 1395, p75FileSize -> 1395, p50FileSize -> 1395, numAddedBytes -> 1395)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
8,2026-04-14T15:59:03.000Z,73664067398892,tangallarohith5@gmail.com,MERGE,"Map(predicate -> [""(id#12083L = id#12103L)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(3132550639151481),e0fe2482-3a3d-4062-b52c-a95bad3184c6,0414-150555-h8g4gkkc-v2n,7,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 2, numTargetBytesAdded -> 2475, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 1, executionTimeMs -> 5087, materializeSourceTimeMs -> 472, numTargetRowsInserted -> 1, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1882, numTargetRowsUpdated -> 1, numOutputRows -> 2, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 2, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2561)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
7,2026-04-14T15:58:41.000Z,73664067398892,tangallarohith5@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(3132550639151481),519c26f7-0a06-4cd7-90ea-88e72fe520a1,0414-150555-h8g4gkkc-v2n,6,SnapshotIsolation,false,"Map(numRemovedFiles -> 3, numRemovedBytes -> 3827, p25FileSize -> 1374, numDeletionVectorsRemoved -> 1, minFileSize -> 1374, numAddedFiles -> 1, maxFileSize -> 1374, p75FileSize -> 1374, p50FileSize -> 1374, numAddedBytes -> 1374)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
6,2026-04-14T15:58:39.000Z,73664067398892,tangallarohith5@gmail.com,UPDATE,"Map(predicate -> [""(id#11725L = 2)""])",null,List(3132550639151481),519c26f7-0a06-4cd7-90ea-88e72fe520a1,0414-150555-h8g4gkkc-v2n,5,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 3059, numDeletionVectorsUpdated -> 0, scanTimeMs -> 1498, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 1239, rewriteTimeMs -> 1559)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
5,2026-04-14T15:53:26.000Z,73664067398892,tangallarohith5@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(3132550639151481),2970fe39-476c-416c-940f-c3d82ea3e780,0414-150555-h8g4gkkc-v2n,4,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1240)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
4,2026-04-14T15:53:25.000Z,73664067398892,tangallarohith5@gmail.com,WRITE,"Map(mode -> Overwrite, statsOnLoad -> false, partitionBy -> [])",null,List(3132550639151481),4fca

In [0]:
df_old = spark.read.format("delta") \
    .option("versionAsOf", 0) \
    .load("/Volumes/workspace/default/day6_session/delta_table_data")
df_old.display()

id,customer_id,product,amount
1,C001,Laptop,50000
2,C002,Mobile,15000
3,C003,Tablet,20000
4,C004,Laptop,55000


In [0]:
delta_table.restoreToVersion(0)

DataFrame[table_size_after_restore: bigint, num_of_files_after_restore: bigint, num_removed_files: bigint, num_restored_files: bigint, removed_files_size: bigint, restored_files_size: bigint]